# 🛡️ Masked IP Detection – Data Collection Notebook

This notebook collects IP intelligence data from public sources such as:
- Tor exit nodes
- Public proxy lists
- VPN provider ASNs
- Datacenter ASNs
- Legitimate IP samples

The collected data is used for training ML models to detect masked or anonymous IP addresses.

In [ ]:
# Install required packages (run in Colab if needed)
# !pip install geoip2 maxminddb python-whois ipwhois requests pandas tqdm

In [ ]:
# Imports
import os
import time
import ipaddress
import requests
import pandas as pd
from datetime import datetime
from tqdm.notebook import tqdm

In [ ]:
# Directory setup
BASE_DIR = 'data/raw'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"Data directory ready: {BASE_DIR}")

## 🔍 IP Data Collector Class

In [ ]:
class IPDataCollector:
    """Collect IP data from public sources"""

    def __init__(self, output_dir):
        self.output_dir = output_dir

    def collect_tor_nodes(self):
        print("Collecting Tor exit nodes...")
        sources = [
            "https://check.torproject.org/exit-addresses",
            "https://www.dan.me.uk/torlist/"
        ]
        tor_ips = set()

        for url in tqdm(sources):
            try:
                r = requests.get(url, timeout=15)
                if r.status_code == 200:
                    for line in r.text.splitlines():
                        if 'ExitAddress' in line:
                            tor_ips.add(line.split()[1])
                        elif line and not line.startswith('#'):
                            ip = line.split()[0]
                            if self._is_valid_ip(ip):
                                tor_ips.add(ip)
                time.sleep(1)
            except Exception as e:
                print(e)

        return pd.DataFrame({
            'ip': list(tor_ips),
            'label': 1,
            'type': 'tor',
            'collected_at': datetime.now()
        })

    def collect_proxy_lists(self):
        print("Collecting proxy IPs...")
        sources = [
            "https://raw.githubusercontent.com/TheSpeedX/PROXY-List/master/http.txt",
            "https://raw.githubusercontent.com/clarketm/proxy-list/master/proxy-list-raw.txt",
            "https://raw.githubusercontent.com/ShiftyTR/Proxy-List/master/proxy.txt"
        ]
        proxy_ips = set()

        for url in tqdm(sources):
            try:
                r = requests.get(url, timeout=15)
                if r.status_code == 200:
                    for line in r.text.splitlines():
                        if ':' in line:
                            ip = line.split(':')[0]
                            if self._is_valid_ip(ip):
                                proxy_ips.add(ip)
                time.sleep(1)
            except Exception as e:
                print(e)

        return pd.DataFrame({
            'ip': list(proxy_ips),
            'label': 1,
            'type': 'proxy',
            'collected_at': datetime.now()
        })

    def collect_vpn_data(self):
        vpn_asns = [
            'AS202795', 'AS43350', 'AS396356', 'AS328543', 'AS62371'
        ]
        return pd.DataFrame({
            'asn': vpn_asns,
            'label': 1,
            'type': 'vpn',
            'collected_at': datetime.now()
        })

    def generate_legitimate_samples(self, n=5000):
        asns = ['AS7922', 'AS20115', 'AS7018', 'AS701', 'AS3356']
        rows = []
        for asn in asns:
            for _ in range(n // len(asns)):
                rows.append({
                    'asn': asn,
                    'label': 0,
                    'type': 'legitimate',
                    'collected_at': datetime.now()
                })
        return pd.DataFrame(rows)

    @staticmethod
    def _is_valid_ip(ip):
        try:
            ipaddress.ip_address(ip)
            return True
        except:
            return False

## 📦 Collect & Save Datasets

In [ ]:
collector = IPDataCollector(BASE_DIR)

datasets = {
    'tor': collector.collect_tor_nodes(),
    'proxy': collector.collect_proxy_lists(),
    'vpn': collector.collect_vpn_data(),
    'legitimate': collector.generate_legitimate_samples()
}

for name, df in datasets.items():
    path = f"{BASE_DIR}/{name}_data.csv"
    df.to_csv(path, index=False)
    print(f"Saved {name}: {len(df)} records → {path}")

all_data = pd.concat(datasets.values(), ignore_index=True)
all_data.to_csv(f"{BASE_DIR}/combined_raw_data.csv", index=False)

print("\nData collection completed successfully!")